In [4]:
import yaml
import pandas as pd
import numpy as np

In [ ]:
# Importing the config file to use values

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

config

{'data': {'input_path': 'data/raw/household_power_consumption.txt',
  'time_interval': None,
  'fill_missing': None}}

In [ ]:
#Since the dataset contains ? and are seperated by ; this needs to be handled to proceed

df = pd.read_csv(config["data"]["input_path"], sep=";", na_values="?", low_memory=False)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   str    
 1   Time                   str    
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), str(2)
memory usage: 142.5 MB


In [ ]:
method = config["data"]["fill_missing"]

if method == "drop":
    df = df.dropna()

elif method == "ffill":
    df = df.fillna(method="ffill")

elif method == "bfill":
    df = df.fillna(method="bfill")

else:
    raise ValueError("Invalid fill_missing option in config")

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

In [9]:
#Its assumed that withNaN means the application is not drawing power or the Sub-meter didnt record anything so the datapoint is removed. Since this is only 1% and will be aggragted into hours its not seen as a problem to remove them

df = df.dropna()

In [ ]:
#To predict the hourly electrict price, the data is converted into the correct time format and then aggreated each x amount of time, Some of the features represent instantaneous measurements so they are averaged instead.

df["datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

df = df.dropna(subset=["datetime"]).set_index("datetime")

cols = [
    "Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity",
    "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

resample_rule = config["data"]["time_interval"]

agg_dict = {
    "Global_active_power": "mean",
    "Global_reactive_power": "mean",
    "Voltage": "mean",
    "Global_intensity": "mean",
    "Sub_metering_1": "sum",
    "Sub_metering_2": "sum",
    "Sub_metering_3": "sum"
}

hourly_df = df.resample(resample_rule).agg(agg_dict)

In [ ]:
#Since time is cyclical the data needs to be converted using cos/sin

hourly_df["hour"] = hourly_df.index.hour
hourly_df["day_of_week"] = hourly_df.index.dayofweek
hourly_df["month"] = hourly_df.index.month

hourly_df["hour_sin"] = np.sin(2 * np.pi * hourly_df["hour"] / 24)
hourly_df["hour_cos"] = np.cos(2 * np.pi * hourly_df["hour"] / 24)

hourly_df["dow_sin"] = np.sin(2 * np.pi * hourly_df["day_of_week"] / 7)
hourly_df["dow_cos"] = np.cos(2 * np.pi * hourly_df["day_of_week"] / 7)

hourly_df["month_sin"] = np.sin(2 * np.pi * hourly_df["month"] / 12)
hourly_df["month_cos"] = np.cos(2 * np.pi * hourly_df["month"] / 12)